In [ ]:
# --- Скрипт для создания индекса ---
import os
import json
import faiss
import numpy as np
from pathlib import Path
import tiktoken
from sentence_transformers import SentenceTransformer
import uuid

# --- 1. Чанкинг с метаданными ---
def chunk_by_tokens(text: str, source: str, max_tokens: int = 256, overlap: int = 32, enc=None):
    enc = enc or tiktoken.get_encoding("cl100k_base")
    tokens = enc.encode(text)
    start = 0
    chunks = []

    # Заголовок для метаданных (берём имя файла без расширения)
    title = Path(source).stem

    while start < len(tokens):
        end = min(start + max_tokens, len(tokens))
        chunk_text = enc.decode(tokens[start:end])
        chunk_id = str(uuid.uuid4())  # уникальный id чанка

        chunks.append({
            "chunk_id": chunk_id,
            "file": source,
            "title": title,
            "chunk": chunk_text,
            "start_token": start,
            "end_token": end,
            "start_char": len(enc.decode(tokens[:start])),
            "end_char": len(enc.decode(tokens[:end]))
        })
        
        if end == len(tokens):
            break
        start = end - overlap

    return chunks

# --- 2. Обработка директории ---
def process_directory(input_dir: str, max_tokens=256, overlap=32):
    enc = tiktoken.get_encoding("cl100k_base")
    all_chunks = []
    for root, _, files in os.walk(input_dir):
        for file in files:
            if Path(file).suffix.lower() != ".md":
                continue
            file_path = os.path.join(root, file)
            text = read_md_file(file_path)
            if not text:
                continue
            file_chunks = chunk_by_tokens(text, file_path, max_tokens, overlap, enc)
            all_chunks.extend(file_chunks)
    return all_chunks

def read_md_file(path: str) -> str:
    with open(path, "r", encoding="utf-8") as f:
        return f.read()

# --- 3. Генерация эмбеддингов ---
def embed_chunks(chunks, model):
    texts = [c["chunk"] for c in chunks]
    vectors = model.encode(texts, convert_to_numpy=True, show_progress_bar=True)
    return vectors

# --- 4. Построение FAISS индекса ---
def build_faiss_index(vectors: np.ndarray):
    dim = vectors.shape[1]
    index = faiss.IndexFlatL2(dim)
    index.add(vectors)
    return index

# --- 5. Сохранение данных ---
def save_faiss_index(index, path="faiss.index"):
    faiss.write_index(index, path)

def save_metadata(chunks, path="metadata.json"):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(chunks, f, ensure_ascii=False, indent=2)

# --- 6. Поиск по FAISS ---
def search(query, model, index, metadata, k=3):
    q_vec = model.encode([query], convert_to_numpy=True)
    D, I = index.search(q_vec, k)
    results = []
    for idx, dist in zip(I[0], D[0]):
        if idx == -1:
            continue
        result = metadata[idx].copy()
        result["score"] = float(dist)
        results.append(result)
    return results

# --- MAIN ---
if __name__ == "__main__":
    input_dir = "../knowledge_base"
    max_tokens = 256
    overlap = 32

    print("📄 Разбиваем файлы на чанки с метаданными...")
    chunks = process_directory(input_dir, max_tokens, overlap)
    print(f"✅ Получено чанков: {len(chunks)}")

    print("🧠 Загружаем модель эмбеддингов...")
    model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

    print("⚡ Генерируем эмбеддинги...")
    vectors = embed_chunks(chunks, model)

    print("📦 Создаём FAISS индекс...")
    index = build_faiss_index(vectors)

    print("💾 Сохраняем индекс и метаданные...")
    save_faiss_index(index, "faiss.index")
    save_metadata(chunks, "metadata.json")

    print("🔍 Пробуем поиск...")
    results = search("Protocol Unit VEX-3", model, index, chunks, k=3)
    for r in results:
        print(f"\n📌 Файл: {r['file']} (Title: {r['title']}, Chunk ID: {r['chunk_id']})")
        print(f"🧭 Диапазон токенов: {r['start_token']}–{r['end_token']}")
        print(f"✍️  Текст:\n{r['chunk'][:300]}...")


📄 Разбиваем файлы на чанки с метаданными...
✅ Получено чанков: 3035
🧠 Загружаем модель эмбеддингов...
⚡ Генерируем эмбеддинги...


Batches:   0%|          | 0/95 [00:00<?, ?it/s]

📦 Создаём FAISS индекс...
💾 Сохраняем индекс и метаданные...
🔍 Пробуем поиск...

📌 Файл: ../knowledge_base_2/PROTOCOL UNIT VEX-3.md (Title: PROTOCOL UNIT VEX-3, Chunk ID: d793a870-a245-4c8b-b74e-94adfecd9b12)
🧭 Диапазон токенов: 24640–24896
✍️  Текст:
 Delegate and subsequently Vanguard Lord. PROTOCOL UNIT VEX-3 managed Lord Calen's Vanguard Construct Servitor spy network. Despite not being able to run or fight, PROTOCOL UNIT VEX-3 bravely stayed behind to hold back The Prime Directive Agent Terex so that Eryk Dane and The spy Construct Servitor ...

📌 Файл: ../knowledge_base_2/PROTOCOL UNIT VEX-3.md (Title: PROTOCOL UNIT VEX-3, Chunk ID: deb4cd40-336e-43d8-928f-7e3cb565d634)
🧭 Диапазон токенов: 23968–24224
✍️  Текст:
 Throughout his operational life, PROTOCOL UNIT VEX-3 worked diligently for several masters including Erynd Korval, Seren Valora, Bail Calen, and Kael Dravorn. As a Liaison Servitor, PROTOCOL UNIT VEX-3 could communicate in over seven million different forms of communicat

In [5]:
# --- Скрипт поиска по индексу ---

import json
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer


FAISS_INDEX_PATH = "faiss.index"
METADATA_PATH = "metadata.json"

# 🧠 Загружаем модель для эмбеддингов
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# 📂 Загружаем FAISS индекс
index = faiss.read_index(FAISS_INDEX_PATH)

# 📄 Загружаем метаданные
with open(METADATA_PATH, "r", encoding="utf-8") as f:
    metadata = json.load(f)

print(f"Загружено {len(metadata)} чанков")

# 🔍 Функция поиска
def search(query, model, index, metadata, k=3):
    # Генерация вектора для запроса
    q_vec = model.encode([query], convert_to_numpy=True)
    D, I = index.search(q_vec, k)
    results = []
    for idx, dist in zip(I[0], D[0]):
        if idx == -1:
            continue
        result = metadata[idx].copy()
        result["score"] = float(dist)
        results.append(result)
    return results

# %%
# ✍️ Вводим запрос
query = "Biography of Kael Dravorn"

# %%
# ⚡ Выполняем поиск
results = search(query, model, index, metadata, k=5)

# %%
# 📌 Вывод результатов
for i, r in enumerate(results, 1):
    print(f"\n=== Результат {i} ===")
    print(f"Файл       : {r['file']}")
    print(f"Заголовок  : {r['title']}")
    print(f"Chunk ID   : {r['chunk_id']}")
    print(f"Диапазон токенов: {r['start_token']}–{r['end_token']}")
    print(f"Семантическая близость: {r['score']:.4f}")
    print(f"Текст:\n{r['chunk'][:500]}...")  # выводим первые 500 символов


FileNotFoundError: [Errno 2] No such file or directory

In [1]:
"""
Проверяет FAISS-индекс и выводит информацию:
- количество векторов (чанков)
- размерность эмбеддингов
"""

import faiss

def check_faiss_index(index_path="faiss.index"):

    # Загружаем индекс
    index = faiss.read_index(index_path)
    
    # Количество векторов
    n_vectors = index.ntotal
    
    # Размерность эмбеддингов
    dim = index.d
    
    print(f"FAISS индекс: {index_path}")
    print(f"Количество векторов (чанков): {n_vectors}")
    print(f"Размерность эмбеддингов: {dim}")
    
    return n_vectors, dim

# Пример использования
n_vectors, dim = check_faiss_index("faiss.index")


RuntimeError: Error in faiss::FileIOReader::FileIOReader(const char *) at /Users/runner/work/faiss-wheels/faiss-wheels/third-party/faiss/faiss/impl/io.cpp:70: Error: 'f' failed: could not open faiss.index for reading: No such file or directory